In [1]:
import pandas as pd

In [2]:
!git clone https://github.com/anycookie112/agile
%cd agile/
!git pull origin main

fatal: destination path 'agile' already exists and is not an empty directory.
/content/agile
From https://github.com/anycookie112/agile
 * branch            main       -> FETCH_HEAD
Already up to date.


# CI/CD Setup with GitHub Actions

In [3]:
%cd /content/agile

!mkdir -p src
!mkdir -p tests
!mkdir -p .github/workflows
!touch src/__init__.py

/content/agile


In [4]:
%%writefile src/remove_duplicates.py
import pandas as pd


def remove_duplicates(input_file, output_file):
    df = pd.read_csv(input_file)
    cleaned_df = df.drop_duplicates()
    cleaned_df.to_csv(output_file, index=False)
    return cleaned_df


if __name__ == "__main__":
    remove_duplicates(
        "data/student_admission.csv",
        "data/processed_dataset.csv"
    )

Overwriting src/remove_duplicates.py


In [5]:
%%writefile tests/test_remove_duplicates.py
import pandas as pd
from src.remove_duplicates import remove_duplicates


def test_remove_duplicates(tmp_path):
    input_file = tmp_path / "input.csv"
    output_file = tmp_path / "output.csv"

    df = pd.DataFrame({
        "Name": ["Ali", "Siti", "Ali"],
        "Age": [20, 21, 20],
        "City": ["KL", "Penang", "KL"]
    })

    df.to_csv(input_file, index=False)

    cleaned_df = remove_duplicates(input_file, output_file)

    assert len(cleaned_df) == 2
    assert cleaned_df.duplicated().sum() == 0
    assert output_file.exists()

Overwriting tests/test_remove_duplicates.py


In [6]:
%%writefile requirements.txt
pandas
pytest

Overwriting requirements.txt


In [7]:
%%writefile .github/workflows/ci.yml
name: Duplicate Removal CI

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  test-duplicate-removal:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run automated tests
        run: pytest

Overwriting .github/workflows/ci.yml


In [8]:
!pip install -r requirements.txt
!pytest

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/agile
plugins: anyio-4.13.0, langsmith-0.7.34, typeguard-4.5.1
collected 0 items / 1 error                                                    

==================================== ERRORS ====================================
_______________ ERROR collecting tests/test_remove_duplicates.py _______________
ImportError while importing test module '/content/agile/tests/test_remove_duplicates.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
/usr/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_remove_duplicates.py:2: in <module>
    from src.remove_duplicates import remove_duplicates
E   ModuleNotFoundError: No module named 'src'
========================

### Step 1: Check status after saving and commit and push code
Make sure you have saved the notebook (File > Save) before running this.

In [9]:
!git status

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.github/
	requirements.txt
	src/
	tests/

nothing added to commit but untracked files present (use "git add" to track)


In [10]:
import os
print(f"Current Directory: {os.getcwd()}")
print("Files in this directory:")
!ls -a
print("\nGit Status Details:")
!git status

Current Directory: /content/agile
Files in this directory:
.   agile.ipynb  .git	  LICENSE	 README.md	   src
..  data	 .github  .pytest_cache  requirements.txt  tests

Git Status Details:
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.github/
	requirements.txt
	src/
	tests/

nothing added to commit but untracked files present (use "git add" to track)


In [11]:
import os

# This copies the current notebook file (where Colab saves by default)
# over to the repository folder where Git can see it.
# Note: You must press Ctrl+S (or File > Save) first for this to work!

source_path = '/content/agile.ipynb'
target_path = '/content/agile/agile.ipynb'

if os.path.exists(source_path):
    !cp {source_path} {target_path}
    print(f"Successfully updated {target_path} with your latest save.")
else:
    print(f"Could not find {source_path}. Make sure you have saved the notebook manually first!")

Could not find /content/agile.ipynb. Make sure you have saved the notebook manually first!


In [12]:
import glob
import os
import shutil

# 1. Search for all .ipynb files on disk
files = glob.glob('/content/*.ipynb') + glob.glob('/content/sample_data/*.ipynb')

# Filter out the one inside the agile folder to find the 'source'
possible_sources = [f for f in files if 'agile/' not in f]

if not possible_sources:
    print("Still couldn't find a notebook file outside the repo folder.")
    print("Try: File -> Save a copy in Drive, or check the file explorer on the left.")
else:
    # Pick the most recently modified one
    latest_file = max(possible_sources, key=os.path.getmtime)
    target = '/content/agile/agile.ipynb'

    shutil.copy(latest_file, target)
    print(f"Found your notebook at: {latest_file}")
    print(f"Successfully copied to: {target}")

    # Verify for Git
    %cd /content/agile
    !git status

Still couldn't find a notebook file outside the repo folder.
Try: File -> Save a copy in Drive, or check the file explorer on the left.


In [13]:
import os
print('Listing all files in /content to find the notebook:')
!find /content -maxdepth 2 -name "*.ipynb"

print('\nChecking root contents:')
!ls -R /content

Listing all files in /content to find the notebook:
/content/agile/agile.ipynb

Checking root contents:
/content:
agile  sample_data

/content/agile:
agile.ipynb  data  LICENSE  README.md  requirements.txt  src  tests

/content/agile/data:
processed_dataset.csv  student_admission.csv

/content/agile/src:
__init__.py  remove_duplicates.py

/content/agile/tests:
__pycache__  test_remove_duplicates.py

/content/agile/tests/__pycache__:
test_remove_duplicates.cpython-312-pytest-8.4.2.pyc

/content/sample_data:
anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md


In [14]:
import json
from google.colab import _message

# Get the current notebook session content
notebook_content = _message.blocking_request('get_ipynb')

# Force write it to the repository path
target_path = '/content/agile/agile.ipynb'
with open(target_path, 'w', encoding='utf-8') as f:
    json.dump(notebook_content['ipynb'], f, indent=1)

print(f"Successfully forced session content into {target_path}")

# Verify with git
%cd /content/agile
!git status

Successfully forced session content into /content/agile/agile.ipynb
/content/agile
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   agile.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.github/
	requirements.txt
	src/
	tests/

no changes added to commit (use "git add" and/or "git commit -a")


In [15]:
# Now that the disk file is actually changed, push it!
!git add agile.ipynb
!git commit -m "Update notebook from Colab session"
!git push origin main

[main 10b4ebc] Update notebook from Colab session
 1 file changed, 59 insertions(+), 148 deletions(-)
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
import json
import os
from google.colab import _message

# 1. Configuration
%cd /content/agile
!git config user.email "joshuatan2000@hotmail.com"
!git config user.name "anycookie112"

# 2. Force session save to disk
target_path = '/content/agile/agile.ipynb'
notebook_content = _message.blocking_request('get_ipynb')
with open(target_path, 'w', encoding='utf-8') as f:
    json.dump(notebook_content['ipynb'], f, indent=1)

# 3. Push to GitHub
!git add agile.ipynb
!git commit -m "Clean update from Colab"
!git push origin main

/content/agile
